# Title

In [1]:
%config InteractiveShell.ast_node_interactivity='last_expr_or_assign'  # always print last expr.
%config InlineBackend.figure_format = 'svg'
%load_ext autoreload
%autoreload 2
%matplotlib inline

In [2]:
import logging
from abc import ABC, abstractmethod
from typing import Any, Literal, Optional, Union
from collections import defaultdict

import torch
import numpy as np
import pandas.api.types
from pandas import DatetimeIndex, Series, Timedelta, Timestamp, DataFrame, Index, MultiIndex, NA
from torch import Tensor

In [3]:
from tsdm.datasets import ETTh1
ds = ETTh1.dataset

,HUFL,HULL,MUFL,MULL,LUFL,LULL,OT
date,,,,,,,
2016-07-01 00:00:00,5.827,2.009,1.599,0.462,4.203,1.340,30.531000
2016-07-01 01:00:00,5.693,2.076,1.492,0.426,4.142,1.371,27.787001
2016-07-01 02:00:00,5.157,1.741,1.279,0.355,3.777,1.218,27.787001
2016-07-01 03:00:00,5.090,1.942,1.279,0.391,3.807,1.279,25.044001
2016-07-01 04:00:00,5.358,1.942,1.492,0.462,3.868,1.279,21.948000
...,...,...,...,...,...,...,...
2018-06-26 15:00:00,-1.674,3.550,-5.615,2.132,3.472,1.523,10.904000
2018-06-26 16:00:00,-5.492,4.287,-9.132,2.274,3.533,1.675,11.044000
2018-06-26 17:00:00,2.813,3.818,-0.817,2.097,3.716,1.523,10.271000


## Standardizer

In [4]:
from tsdm.encoders.modular import Standardizer

encoder = Standardizer()
encoder.fit(ds)
encoded = encoder.encode(ds)

,HUFL,HULL,MUFL,MULL,LUFL,LULL,OT
date,,,,,,,
2016-07-01 00:00:00,-0.219049,-0.114207,-0.395683,-0.231903,0.976355,0.805738,2.008513
2016-07-01 01:00:00,-0.238009,-0.081400,-0.411356,-0.251800,0.923970,0.857445,1.688203
2016-07-01 02:00:00,-0.313849,-0.245432,-0.442557,-0.291043,0.610524,0.602247,1.688203
2016-07-01 03:00:00,-0.323329,-0.147013,-0.442557,-0.271146,0.636286,0.703993,1.368010
2016-07-01 04:00:00,-0.285409,-0.147013,-0.411356,-0.231903,0.688671,0.703993,1.006610
...,...,...,...,...,...,...,...
2018-06-26 15:00:00,-1.280380,0.640341,-1.452403,0.691136,0.348602,1.110975,-0.282568
2018-06-26 16:00:00,-1.820597,1.001211,-1.967580,0.769622,0.400987,1.364505,-0.266225
2018-06-26 17:00:00,-0.645506,0.771566,-0.749583,0.671791,0.558139,1.110975,-0.356458


In [5]:
decoded = encoder.decode(ds)

,HUFL,HULL,MUFL,MULL,LUFL,LULL,OT
date,,,,,,,
2016-07-01 00:00:00,48.557704,6.345190,15.216264,1.717437,7.960340,1.660309,274.874603
2016-07-01 01:00:00,47.610652,6.482023,14.485798,1.652305,7.889307,1.678894,251.367582
2016-07-01 02:00:00,43.822451,5.797858,13.031694,1.523848,7.464275,1.587165,251.367582
2016-07-01 03:00:00,43.348927,6.208357,13.031694,1.588981,7.499209,1.623737,227.869122
2016-07-01 04:00:00,45.243025,6.208357,14.485798,1.717437,7.570242,1.623737,201.346612
...,...,...,...,...,...,...,...
2018-06-26 15:00:00,-4.455923,9.492349,-34.032142,4.738871,7.109110,1.770023,106.735975
2018-06-26 16:00:00,-31.439795,10.997513,-58.041936,4.995783,7.180143,1.861153,107.935308
2018-06-26 17:00:00,27.256134,10.039682,-1.277242,4.675547,7.393242,1.770023,101.313251


## ChainedEncoder

In [6]:
enc = Standardizer() @ Standardizer()

ChainedEncoder[Standardizer([-1]), Standardizer([-1])]

In [7]:
enc.fit(ds)
enc[0].mean, enc[1].mean

(array([ 2.03944528e-17, -6.52622490e-17,  0.00000000e+00, -2.61048996e-17,
        -9.46302610e-17,  7.83146988e-17,  7.50515863e-17]),
 array([ 7.37514133,  2.24224242,  4.30023909,  0.88156774,  3.066062  ,
         0.85693215, 13.32467159]))

In [8]:
encoder = Standardizer()
encoder.fit(ds)
encoded = encoder.encode(ds)

,HUFL,HULL,MUFL,MULL,LUFL,LULL,OT
date,,,,,,,
2016-07-01 00:00:00,-0.219049,-0.114207,-0.395683,-0.231903,0.976355,0.805738,2.008513
2016-07-01 01:00:00,-0.238009,-0.081400,-0.411356,-0.251800,0.923970,0.857445,1.688203
2016-07-01 02:00:00,-0.313849,-0.245432,-0.442557,-0.291043,0.610524,0.602247,1.688203
2016-07-01 03:00:00,-0.323329,-0.147013,-0.442557,-0.271146,0.636286,0.703993,1.368010
2016-07-01 04:00:00,-0.285409,-0.147013,-0.411356,-0.231903,0.688671,0.703993,1.006610
...,...,...,...,...,...,...,...
2018-06-26 15:00:00,-1.280380,0.640341,-1.452403,0.691136,0.348602,1.110975,-0.282568
2018-06-26 16:00:00,-1.820597,1.001211,-1.967580,0.769622,0.400987,1.364505,-0.266225
2018-06-26 17:00:00,-0.645506,0.771566,-0.749583,0.671791,0.558139,1.110975,-0.356458


In [9]:
decoded = encoder.decode(ds)

,HUFL,HULL,MUFL,MULL,LUFL,LULL,OT
date,,,,,,,
2016-07-01 00:00:00,48.557704,6.345190,15.216264,1.717437,7.960340,1.660309,274.874603
2016-07-01 01:00:00,47.610652,6.482023,14.485798,1.652305,7.889307,1.678894,251.367582
2016-07-01 02:00:00,43.822451,5.797858,13.031694,1.523848,7.464275,1.587165,251.367582
2016-07-01 03:00:00,43.348927,6.208357,13.031694,1.588981,7.499209,1.623737,227.869122
2016-07-01 04:00:00,45.243025,6.208357,14.485798,1.717437,7.570242,1.623737,201.346612
...,...,...,...,...,...,...,...
2018-06-26 15:00:00,-4.455923,9.492349,-34.032142,4.738871,7.109110,1.770023,106.735975
2018-06-26 16:00:00,-31.439795,10.997513,-58.041936,4.995783,7.180143,1.861153,107.935308
2018-06-26 17:00:00,27.256134,10.039682,-1.277242,4.675547,7.393242,1.770023,101.313251


## DataFrameEncoder

In [10]:
DataFrame(ds["OT"]).dtypes

OT    float64
dtype: object

In [11]:
from tsdm.encoders.modular import DataFrameEncoder, Standardizer, DateTimeEncoder
from tsdm.datasets import ETTh1

ds = ETTh1.dataset

encoderA = Standardizer()
encoderB = Standardizer()
encoderC = Standardizer()

encoders = {
    "HUFL" : encoderA,
    "HULL" : encoderB,
    "MUFL" : encoderA,
    "MULL" : encoderC,
    "LUFL" : encoderB,
    "LULL" : encoderB,
    "OT"   : encoderB,
}

{'HUFL': Standardizer([-1]),
 'HULL': Standardizer([-1]),
 'MUFL': Standardizer([-1]),
 'MULL': Standardizer([-1]),
 'LUFL': Standardizer([-1]),
 'LULL': Standardizer([-1]),
 'OT': Standardizer([-1])}

In [12]:
e = DataFrameEncoder(encoders, index_encoder=DateTimeEncoder("h"))

col  \
section partition                           
index   NaN                           NaN   
columns 0                          [MULL]   
        1          [HULL, LUFL, LULL, OT]   
        2                    [HUFL, MUFL]   

                                                             encoder  
section partition                                                     
index   NaN        <tsdm.encoders.modular._modular.DateTimeEncode...  
columns 0                                         Standardizer([-1])  
        1                                         Standardizer([-1])  
        2                                         Standardizer([-1])

In [13]:
e.fit(ds)
encoded = e.encode(ds)
decoded = e.decode(encoded)
pandas.testing.assert_frame_equal(ds, decoded)

### DataFrameEncoder Test Implementation

In [14]:
from tsdm.encoders.modular import BaseEncoder

class DataFrameEncoder:
    r"""Combine multiple encoders into a single one.

    It is assumed that the DataFrame Modality doesn't change.
    """

    column_encoder: Union[BaseEncoder, dict[Any, BaseEncoder]]
    r"""Encoders for the columns."""
    index_encoder: Optional[BaseEncoder] = None
    r"""Optional Encoder for the index."""
    colspec: list[str] = None
    r"""The columns-specification of the DataFrame."""
    encode_index: bool
    r"""Whether to encode the index."""
    column_wise: bool
    r"""Whether to encode column-wise"""
    partitions: Optional[dict] = None
    r"""Contains partitions if used column wise"""


    def __init__(
        self,
        encoders: Union[BaseEncoder, dict[Any, BaseEncoder]],
        *,
        index_encoder: Optional[BaseEncoder] = None,
    ):
        r"""Set up the individual encoders.

        Note: the same encoder instance can be used for multiple columns.

        Parameters
        ----------
        encoders
        index_encoder
        """
        super().__init__()
        self.column_encoder = encoders
        self.index_encoder = index_encoder
        self.column_wise: bool = isinstance(self.column_encoder, dict)
        self.encode_index: bool = index_encoder is not None

        if self.encode_index:
            _idxenc_spec = {
                "col": NA,
                "encoder": self.index_encoder,
                "dim_in": NA,
                "dim_out": NA,
            }
            idxenc_spec = DataFrame.from_records(
                _idxenc_spec, index=Index([NA], name="partition")
            )
        else:
            idxenc_spec = DataFrame(
                columns=["col", "encoder", "dim_in", "dim_out"],
                index=Index([], name="partition"),
            )

        if not self.column_wise:
            _colenc_spec = {
                "col": NA,
                "encoder": self.column_encoder,
                "dim_in": NA,
                "dim_out": NA,
            }
            colenc_spec = DataFrame.from_records(
                _colenc_spec, index=Index([0], name="partition")
            )
        else:
            keys = self.column_encoder.keys()
            assert len(set(keys)) == len(keys), f"Some keys are duplicates!"

            _encoders = tuple(set(self.column_encoder.values()))
            encoders = Series(_encoders, name="encoder")
            partitions = Series(range(len(_encoders)), name="partition")

            _columns = defaultdict(list)
            for key, encoder in self.column_encoder.items():
                _columns[encoder].append(key)

            columns = Series(_columns, name="col")

            colenc_spec = DataFrame(encoders, index=partitions)
            colenc_spec = colenc_spec.join(columns, on="encoder")
            colenc_spec["dim_in"] = colenc_spec["col"].apply(len)
            colenc_spec["dim_out"] = pandas.NA

        self.spec = pandas.concat(
            [idxenc_spec, colenc_spec],
            keys=["index", "columns"],
            names=["section", "partition"],
        )

In [15]:
e = DataFrameEncoder(encoders, index_encoder=Standardizer())
e.spec

col dim_in dim_out             encoder
section partition                                                           
index   NaN                           NaN   <NA>     NaN  Standardizer([-1])
columns 0                          [MULL]      1     NaN  Standardizer([-1])
        1          [HULL, LUFL, LULL, OT]      4     NaN  Standardizer([-1])
        2                    [HUFL, MUFL]      2     NaN  Standardizer([-1])

In [16]:
e.spec.loc["index","dim_out"] = 3
torch.randn(3,4,5)[..., e.spec.loc["index", "dim_out"].item()].shape

torch.Size([3, 4])

In [17]:
e.spec.loc["index", "dim_out"].item()

3

In [18]:
torch.Tensor(ds.values).nanmean(axis=None)

tensor(4.5781)

In [19]:
torch.mean(torch.tensor([float('nan'), 2]))

tensor(nan)

In [20]:
np.nanstd

<function numpy.nanstd(a, axis=None, dtype=None, out=None, ddof=0, keepdims=<no value>)>

In [21]:
torch.nanmean(torch.Tensor(ds.values), dim=None)

tensor(4.5781)

In [22]:
torch.Tensor(ds.values).nanmean(dim=None)

tensor(4.5781)

In [23]:
ds["OT"].mean()

13.324671589881694

In [24]:
indexenc = DataFrame.from_records({"Encoders": object(), "dim_in":1, "dim_out": 5}, index=Index([0]))

,Encoders,dim_in,dim_out
0,<object object at 0x7f19bd315540>,1,5


In [25]:
DataFrame.from_dict({"Encoders": object(), "dim_in":1, "dim_out": 5}, orient='index').T

,Encoders,dim_in,dim_out
0,<object object at 0x7f19bd315590>,1,5


In [26]:
pandas.concat([indexenc, e.spec], keys=['index', 'columns'], names=["section", "partiton"])

Encoders dim_in dim_out  \
section partiton                                                         
index   0             <object object at 0x7f19bd315540>      1       5   
columns (index, nan)                                NaN   <NA>       3   
        (columns, 0)                                NaN      1     NaN   
        (columns, 1)                                NaN      4     NaN   
        (columns, 2)                                NaN      2     NaN   

                                         col             encoder  
section partiton                                                  
index   0                                NaN                 NaN  
columns (index, nan)                     NaN  Standardizer([-1])  
        (columns, 0)                  [MULL]  Standardizer([-1])  
        (columns, 1)  [HULL, LUFL, LULL, OT]  Standardizer([-1])  
        (columns, 2)            [HUFL, MUFL]  Standardizer([-1])

In [27]:
idx = MultiIndex(
    levels=[["index", "columns"],[]],
    codes=[[],[]], 
    names=["section", "partition"]
)

MultiIndex([], names=['section', 'partition'])

In [28]:
spec = DataFrame(
    columns=["col", "encoder", "dim_in", "dim_out"],
    index=Index([], name="partition")
    # index=MultiIndex(
    #     levels=[["index", "columns"], []],
    #     codes=[[], []],
    #     names=["section", "partition"],
    # ),
)

,col,encoder,dim_in,dim_out
partition,,,,


In [29]:
_index_encoder_spec = {
    "col": NA,
    "encoder": object(),
    "dim_in": NA,
    "dim_out": NA,
}
index_encoder_spec = DataFrame.from_records(
    _index_encoder_spec, index=Index([0], name="partition")
)

,col,dim_in,dim_out,encoder
partition,,,,
0,<NA>,<NA>,<NA>,<object object at 0x7f19bd315780>


## Concat Encoder

## Time2Float

## DateTimeEncoder

### Combined

In [85]:
from tsdm.encoders.modular import DateTimeEncoder, DataFrameEncoder, IdentityEncoder, TensorEncoder, Standardizer

In [105]:
df_encoder = DataFrameEncoder(
    Standardizer(), 
    index_encoder=DateTimeEncoder()
)

Tensorizer = TensorEncoder()

encoder = TensorEncoder() @ df_encoder

ChainedEncoder[<tsdm.encoders.modular._modular.TensorEncoder object at 0x7f19b2715130>,                     col                                            encoder
section partition                                                         
index   NaN        <NA>  <tsdm.encoders.modular._modular.DateTimeEncode...
columns 0          <NA>                                 Standardizer([-1])]

In [106]:
from tsdm.datasets import ETTh1
ds = ETTh1.dataset

,HUFL,HULL,MUFL,MULL,LUFL,LULL,OT
date,,,,,,,
2016-07-01 00:00:00,5.827,2.009,1.599,0.462,4.203,1.340,30.531000
2016-07-01 01:00:00,5.693,2.076,1.492,0.426,4.142,1.371,27.787001
2016-07-01 02:00:00,5.157,1.741,1.279,0.355,3.777,1.218,27.787001
2016-07-01 03:00:00,5.090,1.942,1.279,0.391,3.807,1.279,25.044001
2016-07-01 04:00:00,5.358,1.942,1.492,0.462,3.868,1.279,21.948000
...,...,...,...,...,...,...,...
2018-06-26 15:00:00,-1.674,3.550,-5.615,2.132,3.472,1.523,10.904000
2018-06-26 16:00:00,-5.492,4.287,-9.132,2.274,3.533,1.675,11.044000
2018-06-26 17:00:00,2.813,3.818,-0.817,2.097,3.716,1.523,10.271000


In [118]:
encoder.fit(ds)

In [119]:
encoded = encoder.encode(ds)

(tensor([0.0000e+00, 3.6000e+03, 7.2000e+03,  ..., 6.2701e+07, 6.2705e+07,
         6.2708e+07], device='cuda:0'),
 tensor([[-0.2190, -0.1142, -0.3957,  ...,  0.9764,  0.8057,  2.0085],
         [-0.2380, -0.0814, -0.4114,  ...,  0.9240,  0.8574,  1.6882],
         [-0.3138, -0.2454, -0.4426,  ...,  0.6105,  0.6022,  1.6882],
         ...,
         [-0.6455,  0.7716, -0.7496,  ...,  0.5581,  1.1110, -0.3565],
         [ 0.2643,  0.7716,  0.1716,  ...,  0.5058,  0.9592, -0.4140],
         [ 0.3875,  0.6403,  0.2758,  ...,  0.5581,  1.0092, -0.4386]],
        device='cuda:0'))

In [120]:
decoded = encoder.decode(encoded)

,HUFL,HULL,MUFL,MULL,LUFL,LULL,OT
date,,,,,,,
2016-07-01 00:00:00,5.827,2.009,1.599,0.462,4.203,1.340,30.531001
2016-07-01 01:00:00,5.693,2.076,1.492,0.426,4.142,1.371,27.787001
2016-07-01 02:00:00,5.157,1.741,1.279,0.355,3.777,1.218,27.787001
2016-07-01 03:00:00,5.090,1.942,1.279,0.391,3.807,1.279,25.044001
2016-07-01 04:00:00,5.358,1.942,1.492,0.462,3.868,1.279,21.948000
...,...,...,...,...,...,...,...
2018-06-26 15:00:00,-1.674,3.550,-5.615,2.132,3.472,1.523,10.904000
2018-06-26 16:00:00,-5.492,4.287,-9.132,2.274,3.533,1.675,11.044000
2018-06-26 17:00:00,2.813,3.818,-0.817,2.097,3.716,1.523,10.271000


In [125]:
modified = encoded[0]*2 , encoded[1] + 1
encoder.decode(modified)

,HUFL,HULL,MUFL,MULL,LUFL,LULL,OT
date,,,,,,,
2016-07-01 00:00:00,19.962083,6.093567,15.252565,4.080483,6.531945,2.539069,47.664401
2016-07-01 02:00:00,19.828082,6.160567,15.145564,4.044483,6.470945,2.570069,44.920401
2016-07-01 04:00:00,19.292082,5.825567,14.932564,3.973483,6.105945,2.417069,44.920401
2016-07-01 06:00:00,19.225083,6.026567,14.932564,4.009483,6.135945,2.478069,42.177401
2016-07-01 08:00:00,19.493082,6.026567,15.145564,4.080483,6.196945,2.478069,39.081402
...,...,...,...,...,...,...,...
2020-06-21 06:00:00,12.461082,7.634567,8.038565,5.750483,5.800945,2.722069,28.037401
2020-06-21 08:00:00,8.643082,8.371568,4.521564,5.892483,5.861945,2.874069,28.177401
2020-06-21 10:00:00,16.948082,7.902567,12.836564,5.715483,6.044945,2.722069,27.404401
